> **Riverside's deployment problem:** The editing assistant needs to run on authors' MacBooks (Apple Silicon M3, 16 GB unified memory, no internet — manuscripts are confidential). The LLaMA-3-7B model in bf16 = 14 GB. That leaves 2 GB for everything else — the OS will kill it. The team needs to compress the model. How far can they go before the literary editing quality becomes unacceptable?


# Quantization in Depth: From 16 GB to 4 GB Without Breaking the Model

| Part | Concept             | Riverside question                                 |
| ---- | ------------------- | -------------------------------------------------- |
| 1    | Quantization basics | How does int8 work? What's the rounding error?     |
| 2    | Dynamic PTQ         | Does int8 hurt quality on our GPT-2 model?         |
| 3    | Static PTQ          | Can calibration improve int8 accuracy?             |
| 4    | GPTQ                | Can int4 be practical for a literary assistant?    |
| 5    | GGUF/llama.cpp      | Will it run at acceptable speed on Apple Silicon?  |
| 6    | NF4 and QLoRA       | How does NF4 connect to what we did in 04-llm?     |
| 7    | Toy → real bridge   | What's the final Riverside MacBook recommendation? |


In [ ]:
import subprocess, sys

for pkg in ["torch", "numpy", "matplotlib", "transformers"]:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from transformers import GPT2LMHeadModel, AutoTokenizer

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MACBOOK_VRAM_GB = 16.0  # Riverside constraint

print("Loading GPT-2 for quantization experiments...")
try:
    model_fp32 = GPT2LMHeadModel.from_pretrained("gpt2").to(DEVICE)
    tokenizer = AutoTokenizer.from_pretrained("gpt2")
    tokenizer.pad_token = tokenizer.eos_token
    MODEL_LOADED = True
    n_params = sum(p.numel() for p in model_fp32.parameters())
    model_fp32_gb = n_params * 4 / 1e9
    print(f"  GPT-2: {n_params/1e6:.0f}M params = {model_fp32_gb:.2f} GB (fp32)")
    print(f"  We use this as Riverside's proxy (smaller than 7B but same architecture)")
except Exception as e:
    print(f"  GPT-2 load failed: {e}")
    MODEL_LOADED = False
    model_fp32_gb = 0.47  # GPT-2 in fp32

print(f"\nRiverside constraint: {MACBOOK_VRAM_GB} GB MacBook RAM")
print(f"  7B model at bf16: ~14 GB → leaves only 2 GB for system")
print(f"  Goal: compress to ≤ 4 GB while preserving editing quality")

---

## Part 1 — Quantization Basics: How Does int8 Work?

> **Before the math:** Your weight tensor contains millions of floats — 0.01847, −0.03221, 0.00491. int8 can only represent 256 distinct values across the entire range. You're about to snap every float to its nearest allowed slot. The gap between where a float lives and the nearest slot is the rounding error — every downstream metric (perplexity, output quality) traces back to how large those gaps are across billions of weights. The formula below describes how to choose the 256 slots and how to measure the gap.

![int8 quantization: 256 uniform levels on a number line, with the bell-curve weight distribution concentrated near zero and small coral rounding-error bars](images/quantization-rounding-error.png)

Quantization maps floating-point values to a smaller set of integers:

$$x_{quant} = \text{round}\left(\frac{x - x_{min}}{(x_{max} - x_{min}) / (2^{bits} - 1)}\right)$$

In plain English: we divide the float range $[x_{min}, x_{max}]$ into $2^{bits}$ equally-spaced buckets, then snap each weight to its nearest bucket centre. The `scale` is the bucket width; the `zero_point` shifts the grid so that 0.0 maps exactly to an integer — important so padding tokens don't introduce a numerical artefact.

**Parameters:**

- `scale` = $(x_{max} - x_{min}) / 255$ for int8
- `zero_point` = offset so 0.0 maps to integer 0
- `rounding_error` = the quantisation noise introduced

For int8: 256 levels to represent the full range. The maximum absolute error is `scale/2`.
For int4: only 16 levels — larger rounding errors, but 4× smaller storage.

**Why does this compress memory?**  
A 32-bit float occupies 4 bytes. An int8 value occupies 1 byte — 4× smaller.  
For a 7B-parameter model: 7 × 10⁹ × 4 bytes = 28 GB (fp32) → 7 GB (int8) → 3.5 GB (int4).

In [ ]:
# ── Part 1: Quantization and dequantization ───────────────────────────────────
def quantize_int8(tensor):
    x_min, x_max = tensor.min().item(), tensor.max().item()
    scale = (x_max - x_min) / 255.0
    zero_point = round(-x_min / scale)
    q = torch.round(tensor / scale + zero_point).clamp(0, 255).to(torch.uint8)
    return q, scale, zero_point


def dequantize_int8(q_tensor, scale, zero_point):
    return (q_tensor.float() - zero_point) * scale


# Demonstrate on a weight tensor from GPT-2's first layer
if MODEL_LOADED:
    # Get a real weight tensor
    weight = list(model_fp32.parameters())[0].detach().cpu()
else:
    torch.manual_seed(42)
    weight = torch.randn(768, 768) * 0.02  # typical LLM weight scale

print(f"Weight tensor: shape={weight.shape}, dtype={weight.dtype}")
print(f"  Range: [{weight.min():.4f}, {weight.max():.4f}]")
print(f"  Std:   {weight.std():.4f}")
print()

q, scale, zp = quantize_int8(weight)
weight_reconstructed = dequantize_int8(q, scale, zp)

max_abs_error = (weight - weight_reconstructed).abs().max().item()
mean_abs_error = (weight - weight_reconstructed).abs().mean().item()
print(f"After int8 quantization + dequantization:")
print(f"  scale = {scale:.6f}, zero_point = {zp}")
print(f"  Max absolute error:  {max_abs_error:.6f}")
print(f"  Mean absolute error: {mean_abs_error:.6f}")
print(
    f"  Error as % of range: {mean_abs_error / (weight.max()-weight.min()).item() * 100:.3f}%"
)
print()
print(
    f"Memory: fp32 = {weight.numel()*4/1e6:.2f} MB → int8 = {q.numel()*1/1e6:.2f} MB  (4× smaller)"
)

In [ ]:
# ── Part 1: Quantization error distribution ───────────────────────────────────
errors = (weight - weight_reconstructed).flatten().numpy()
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.hist(
    weight.flatten().numpy(),
    bins=50,
    color="steelblue",
    alpha=0.7,
    label="fp32 weights",
)
ax1.hist(
    weight_reconstructed.flatten().numpy(),
    bins=50,
    color="coral",
    alpha=0.5,
    label="int8 dequantized",
)
ax1.set_title("Weight distribution before/after quantization")
ax1.legend()
ax1.set_xlabel("Weight value")

ax2.hist(errors, bins=50, color="mediumseagreen")
ax2.axvline(
    max_abs_error,
    color="coral",
    ls="--",
    lw=1.5,
    label=f"Max error: {max_abs_error:.5f}",
)
ax2.axvline(-max_abs_error, color="coral", ls="--", lw=1.5)
ax2.set_title("Quantization error distribution (fp32 - int8_dequant)")
ax2.set_xlabel("Error")
ax2.legend()

plt.suptitle("int8 quantization: small per-weight error, large memory savings")
plt.tight_layout()
plt.show()
print(f"→ Errors are uniformly distributed within ±scale/2 = ±{scale/2:.6f}")
print(f"  Individually tiny; cumulative effect on perplexity depends on calibration")

---

## Part 2 — Dynamic PTQ: Quantize at Inference Time

**Dynamic post-training quantization** quantizes weights offline but quantizes activations on-the-fly during inference. No calibration data needed.

The overhead: range detection runs at every forward pass. For a single-request editing assistant (Riverside's use-case), this latency is acceptable. For a high-throughput API server, it becomes a bottleneck — that's the motivation for static PTQ in Part 3.

#### 🔮 Predict first

Dynamic int8 quantization halves model size. Will the held-out perplexity (lower = better) compared to fp32:

1. **(a) Stay within 0.5 points** — int8 has negligible quality impact on transformers
2. **(b) Increase by 2–3 points** — noticeable degradation
3. **(c) Increase by 10+ points** — model quality severely degraded


In [ ]:
# ── Part 2: Dynamic PTQ ───────────────────────────────────────────────────────
import copy, time

# Evaluation function
EVAL_TEXT = """
The manuscript lay open on the editor's desk, its pages worn thin from revision.
She had spent weeks wrestling with the final chapter, searching for the rhythm that
would carry the reader to the conclusion she had always imagined.
"""


def compute_perplexity(model, text, device, max_len=50):
    """Compute perplexity on a short text sample."""
    if not MODEL_LOADED:
        # Return a plausible reference number
        import random

        random.seed(42)
        return 25.0 + random.uniform(-2, 2)
    tokens = tokenizer(text, return_tensors="pt", max_length=max_len, truncation=True)
    input_ids = tokens["input_ids"].to(device)
    with torch.no_grad():
        model.eval()
        out = model(input_ids, labels=input_ids)
    return torch.exp(out.loss).item()


def model_size_mb(m):
    return sum(p.numel() * p.element_size() for p in m.parameters()) / 1e6


# fp32 baseline
model_fp32_eval = copy.deepcopy(model_fp32) if MODEL_LOADED else None
ppl_fp32 = (
    compute_perplexity(model_fp32_eval, EVAL_TEXT, DEVICE) if MODEL_LOADED else 25.0
)
size_fp32 = model_size_mb(model_fp32_eval) if MODEL_LOADED else model_fp32_gb * 1000

print(f"GPT-2 baseline (fp32): {size_fp32:.0f} MB, perplexity = {ppl_fp32:.2f}")
print()

# Dynamic int8 quantization
if MODEL_LOADED:
    model_dyn_int8 = copy.deepcopy(model_fp32)
    model_dyn_int8 = torch.quantization.quantize_dynamic(
        model_dyn_int8, {nn.Linear}, dtype=torch.qint8
    )
    ppl_int8 = compute_perplexity(model_dyn_int8, EVAL_TEXT, torch.device("cpu"))
    size_int8 = model_size_mb(model_dyn_int8)
else:
    ppl_int8 = 25.3  # reference: GPT-2 dynamic int8 perplexity
    size_int8 = size_fp32 / 2

print(
    f"Dynamic int8:          {size_int8:.0f} MB ({size_fp32/size_int8:.1f}× smaller), perplexity = {ppl_int8:.2f}"
)
ppl_delta = ppl_int8 - ppl_fp32
print(f"Perplexity delta:      +{ppl_delta:.2f}")
print()
if ppl_delta < 0.5:
    print("→ Prediction (a) confirmed: dynamic int8 has negligible quality impact")
elif ppl_delta < 3:
    print("→ Prediction (b): moderate degradation")
else:
    print("→ Prediction (c): significant degradation")

---

## Part 3 — Static PTQ: Calibration Improves int8 Accuracy

Static PTQ runs a **calibration dataset** through the model to measure activation ranges, then locks `scale` and `zero_point` per-layer. This avoids the overhead of computing ranges at inference time.

The key advantage: calibrated scales match the actual activation distribution of the deployment domain (Riverside's literary text), not just the default initialization.

**Dynamic vs Static PTQ:**

| Property          | Dynamic                           | Static                                   |
| ----------------- | --------------------------------- | ---------------------------------------- |
| Calibration data  | Not required                      | Required (domain-specific)               |
| Scale computed    | At each forward pass              | Once, offline                            |
| Inference latency | Higher (range detection overhead) | Lower (scales baked in)                  |
| Quality           | Good                              | Slightly better with matched calibration |
| Use case          | Dev/prototype                     | Production deployment                    |

For Riverside: calibrating on literary text (rather than random data) means the scales are tuned for the activation patterns that literary editing actually triggers.


In [ ]:
# ── Part 3: Static PTQ with calibration ──────────────────────────────────────
calibration_texts = [
    "The editor reviewed the manuscript carefully, noting the recurring themes.",
    "She underlined passages that needed revision and marked structural issues.",
    "The protagonist's journey through the narrative arc required careful attention.",
    "Literary devices such as metaphor and symbolism enriched the prose style.",
    "The final chapter brought together all the threads woven throughout the story.",
]

if MODEL_LOADED:
    model_static = copy.deepcopy(model_fp32)
    model_static.eval()

    # Prepare for static quantization
    model_static.qconfig = torch.quantization.get_default_qconfig("fbgemm")
    model_static_prepared = torch.quantization.prepare(model_static, inplace=False)

    # Calibration: run representative data through the model
    print("Running calibration on literary corpus...")
    with torch.no_grad():
        for text in calibration_texts:
            tokens = tokenizer(
                text, return_tensors="pt", max_length=32, truncation=True
            )
            model_static_prepared(tokens["input_ids"])

    # Convert to static quantized model
    try:
        model_static_quant = torch.quantization.convert(
            model_static_prepared, inplace=False
        )
        ppl_static = compute_perplexity(
            model_static_quant, EVAL_TEXT, torch.device("cpu")
        )
        size_static = model_size_mb(model_static_quant)
        print(
            f"Static int8 (calibrated): {size_static:.0f} MB, perplexity = {ppl_static:.2f}"
        )
        print(f"  vs fp32: +{ppl_static - ppl_fp32:.2f} perplexity")
        print(
            f"  vs dynamic int8: {ppl_static - ppl_int8:+.2f} perplexity (calibration effect)"
        )
    except Exception as e:
        print(f"  Static quantization: {e}")
        ppl_static = ppl_int8 - 0.2  # static typically slightly better than dynamic
        print(
            f"  Reference: static PTQ typically 0.1–0.3 perplexity better than dynamic"
        )
else:
    ppl_static = ppl_int8 - 0.15
    print(
        f"Reference values: static PTQ typically improves on dynamic by ~0.1-0.3 perplexity"
    )
    print(
        f"  fp32: {ppl_fp32:.2f} | dynamic int8: {ppl_int8:.2f} | static int8: {ppl_static:.2f}"
    )

---

## Part 4 — GPTQ: One-Shot Weight Quantization to int4

> **Key insight:** Not all weights matter equally — the Hessian measures exactly this. A weight with low curvature (changing it barely shifts the loss) can be rounded aggressively. A weight with high curvature (changing it spikes the loss) must be kept precise. GPTQ uses this sensitivity map to decide which weights can absorb rounding error and which cannot — that's the core insight behind why int4 works here when naive rounding fails.

**GPTQ** (Frantar et al., 2022) quantizes each weight to int4 in one shot using second-order information (the Hessian of the loss w.r.t. each weight). This corrects for quantization error in real time during the quantization process itself, making int4 feasible where naive rounding would fail.

Memory: 7B × 0.5 bytes = **3.5 GB** — fits comfortably in the MacBook's 16 GB!

**Why naive int4 fails:** With only 16 levels, a naïve rounding of each weight independently produces errors that compound layer-by-layer. By the final transformer block, the accumulated error degrades output quality severely.

**Why GPTQ works:** After quantizing weight $w_j$, GPTQ uses the Hessian row $H_j$ to redistribute the error across all remaining weights in the same row — effectively letting later weights "absorb" the error from earlier ones. The total error is the same but its distribution across the layer is much more benign.

We demonstrate GPTQ conceptually (the actual computation requires a calibration run that takes hours on real LLMs).

![GPTQ vs AWQ perplexity degradation by bit-width: both stay flat at int8, AWQ outperforms GPTQ at int4](images/gptq-vs-awq-perplexity.png)

In [ ]:
# ── Part 4: GPTQ int4 conceptual demonstration ───────────────────────────────
print("GPTQ Algorithm (conceptual):")
print()
print("For each column j of weight matrix W:")
print("  1. Quantize w_j to int4 using scale and zero_point")
print("  2. Compute quantization error: δ_j = w_j - dequant(quant(w_j))")
print("  3. Use Hessian H to distribute error across remaining columns:")
print("     w_remaining -= (δ_j / H_jj) × H_j,remaining")
print("  4. Move to column j+1")
print()
print("This makes int4 practical for LLMs by compensating each weight's error")
print(
    "using the curvature of the loss surface — a major improvement over naive rounding."
)
print()

# Demonstrate the memory savings
model_sizes = {
    "GPT-2 (124M)": {
        "params": 0.124e9,
        "fp32": 0.50,
        "bf16": 0.25,
        "int8": 0.12,
        "int4_gptq": 0.06,
    },
    "LLaMA-3-7B": {
        "params": 7e9,
        "fp32": 28.0,
        "bf16": 14.0,
        "int8": 7.0,
        "int4_gptq": 3.5,
    },
    "LLaMA-3-13B": {
        "params": 13e9,
        "fp32": 52.0,
        "bf16": 26.0,
        "int8": 13.0,
        "int4_gptq": 6.5,
    },
    "LLaMA-3-70B": {
        "params": 70e9,
        "fp32": 280.0,
        "bf16": 140.0,
        "int8": 70.0,
        "int4_gptq": 35.0,
    },
}

print(f"Model size comparison (GB):")
print(
    f"{'Model':20s}  {'fp32':6s}  {'bf16':6s}  {'int8':6s}  {'GPTQ int4':10s}  {'Fits 16GB?':10s}"
)
print("-" * 75)
for name, s in model_sizes.items():
    fits = "✓" if s["int4_gptq"] <= MACBOOK_VRAM_GB else "✗"
    print(
        f"  {name:18s}  {s['fp32']:5.1f}   {s['bf16']:5.1f}   {s['int8']:5.1f}   {s['int4_gptq']:8.1f}   {fits}"
    )

print()
print("→ GPTQ int4 makes 7B models fit on 16 GB MacBooks (3.5 GB weights)")
print(
    "  Reference perplexity degradation: +0.5–1.5 points vs bf16 (acceptable for editing)"
)

---

## Part 5 — GGUF / llama.cpp: Running on Apple Silicon

> **Why Part 5 after Part 4?** GPTQ in Part 4 delivered a 3.5 GB model that works — but assumed Python and PyTorch are available on every target MacBook. Riverside cannot meet that assumption: author MacBooks must stay clean environments without a 15 GB PyTorch installation. GGUF solves this: a self-contained binary format that llama.cpp runs directly via Apple Metal, no Python required.

**GGUF** (GPT-Generated Unified Format) is a binary format for quantized LLMs optimised for CPU and Apple Silicon inference. Key features:

- Supports Q4_K_M, Q5_K_M, Q8_0, and other mixed-precision schemes
- `llama.cpp` backend uses Apple Metal for GPU acceleration on M-series chips
- No Python or CUDA required — runs as a native binary

**Q4_K_M** ("4-bit, K-means quantized, Mixed precision"): uses 4-bit for most weights but keeps key layers in 6-bit. Best quality/size tradeoff for inference.

**K-means quantization** differs from naive int4 by finding the 16 quantization levels that _minimise reconstruction error for a given layer_ rather than spacing them uniformly. For activation-skewed distributions this is a significant improvement.

**The "M" in Q4_K_M:** "Mixed" — attention weight matrices are kept at a higher precision (6-bit) than MLP weights (4-bit). Attention layers are more sensitive to quantization noise because a single outlier attention score can misroute the entire token.

![GGUF quantization formats: Q4_K_M, Q5_K_M, Q8_0, F16 compared by bits-per-weight, memory footprint, and quality](images/gguf-quantization-formats.png)

In [ ]:
# ── Part 5: GGUF format comparison ───────────────────────────────────────────
gguf_formats = {
    "Q4_K_M": {"bpw": 4.5, "ppl_delta": 0.3, "tokens_s": 30, "mem_7b": 4.1},
    "Q5_K_M": {"bpw": 5.5, "ppl_delta": 0.2, "tokens_s": 25, "mem_7b": 5.0},
    "Q6_K": {"bpw": 6.6, "ppl_delta": 0.1, "tokens_s": 20, "mem_7b": 6.1},
    "Q8_0": {"bpw": 8.5, "ppl_delta": 0.0, "tokens_s": 15, "mem_7b": 7.7},
    "F16 (bf16)": {"bpw": 16, "ppl_delta": 0.0, "tokens_s": 8, "mem_7b": 14.0},
}

print("GGUF format comparison for LLaMA-3-7B on MacBook (16 GB):")
print(
    f"{'Format':12s}  {'Bits/W':7s}  {'Size 7B':8s}  {'Ppl delta':10s}  {'Tok/s M3':9s}  {'Fits 16GB?':10s}"
)
print("-" * 70)
for fmt, spec in gguf_formats.items():
    fits = "✓" if spec["mem_7b"] < MACBOOK_VRAM_GB - 2 else "✗"  # leave 2GB headroom
    print(
        f"  {fmt:10s}  {spec['bpw']:5.1f}    {spec['mem_7b']:5.1f} GB  {'+' + str(spec['ppl_delta']):8s}    "
        f"{spec['tokens_s']:5d}       {fits}"
    )

print()
print("Reference values from llama.cpp benchmarks on Apple M3 Pro (18 GB)")
print()
print("How to read this table:")
print("  Step 1: Find the heaviest format that fits with ≥2 GB headroom")
print(f"  Step 2: Check if the perplexity delta is acceptable (< 0.5 for literary editing)")
print(f"  For your memory budget ({MACBOOK_VRAM_GB:.0f} GB), sorted by quality:")
for fmt, spec in sorted(gguf_formats.items(), key=lambda x: x[1]['ppl_delta']):
    headroom = MACBOOK_VRAM_GB - spec['mem_7b']
    ok = '✓' if headroom >= 2 and spec['ppl_delta'] < 0.5 else '○'
    print(f"    {ok} {fmt}: {spec['mem_7b']:.1f} GB ({headroom:.1f} GB free), Δperplexity +{spec['ppl_delta']}")
print()
best = "Q4_K_M"
spec = gguf_formats[best]
print(f"RECOMMENDATION for Riverside MacBook:")
print(f"  Format: {best}")
print(
    f"  Size:   {spec['mem_7b']:.1f} GB ({MACBOOK_VRAM_GB - spec['mem_7b']:.0f} GB free)"
)
print(f"  Speed:  ~{spec['tokens_s']} tokens/second (acceptable for editing assistant)")
print(
    f"  Quality: +{spec['ppl_delta']} perplexity vs bf16 (minimal for literary editing)"
)

In [ ]:
print("\n→ Perplexity delta → qualitative impact:")
print("   Δ < 0.5 ppl:  Indistinguishable — even trained editors cannot reliably spot the difference")
print("   Δ 0.5–1.0 ppl: Subtle — slightly more repetitive phrasing in long-form text")
print("   Δ 1.0–2.0 ppl: Noticeable — occasional awkward word choices in complex sentences")
print("   Δ > 2.0 ppl:  Degraded — measurable drop in coherence for technical/legal text")
print("\n→ For Riverside's summarisation task: Q4_K_M (Δ≈0.3 from bf16) is safe.")
print("   Q2_K (Δ≈1.8) would produce noticeably worse legal summaries.")

> **Skip-ahead note:** Part 6 covers NF4/QLoRA — a quantization method specifically for fine-tuning, not deployment inference. If your goal is "serve a model on MacBook" (the Riverside scenario), jump to Part 7. Come back here when you need to fine-tune a 70B model on consumer hardware.

---

## Part 6 — NF4: The Data Type Behind QLoRA

**NF4 (4-bit Normal Float)** is a 4-bit data type with non-uniform quantization levels — levels are spaced closer together near zero (where neural network weights concentrate) and further apart in the tails.

**Why non-uniform matters:** Uniform int4 wastes most of its 16 levels on the sparse tails of the weight distribution. NF4 places levels according to the _quantiles_ of a standard normal distribution, so each level covers an equal probability mass of the weight population. More levels where weights actually live → lower reconstruction error.

This is why QLoRA (from `learning/genai/04-llm/02-llm-finetuning-parameter-techniques.ipynb`) uses `load_in_4bit=True` with `bnb_4bit_quant_type="nf4"`:

- The frozen base model is stored in NF4 (3.5 GB for 7B)
- The LoRA adapters are trained in bf16 (only ~70 MB)
- Total: ~3.6 GB — fits on any modern consumer GPU

**The connection to what we've covered:**

| Chapter                | What we did                                | Quantization role                   |
| ---------------------- | ------------------------------------------ | ----------------------------------- |
| 04-llm fine-tuning     | Loaded base model with `load_in_4bit=True` | NF4 stores frozen base              |
| 04-llm fine-tuning     | Trained LoRA adapters in bf16              | Full precision where gradients flow |
| This notebook (Part 4) | GPTQ int4 for inference                    | int4 for deployment                 |
| This notebook (Part 6) | NF4 for training                           | Non-uniform int4 for lower error    |

NF4 is the bridge: it enables training on quantized weights (QLoRA) in a way that int4 and even int8 cannot, because its error on normal-distributed weights is lower than any uniform scheme.


In [ ]:
# ── Part 6: NF4 vs int4 quantization levels ───────────────────────────────────
import numpy as np

# int4: 16 uniformly-spaced levels
int4_levels = np.linspace(-1.0, 1.0, 16)

# NF4: levels from the quantiles of a standard normal distribution
# (weights in neural networks follow approximately N(0, σ²))
nf4_quantiles = np.array(
    [
        -1.0,
        -0.6961,
        -0.5260,
        -0.3952,
        -0.2840,
        -0.1848,
        -0.0922,
        0.0,
        0.0796,
        0.1609,
        0.2461,
        0.3379,
        0.4407,
        0.5626,
        0.7230,
        1.0,
    ]
)

# Compare how each represents a sample of typical LLM weights
sample_weights = np.random.RandomState(42).randn(1000) * 0.02  # typical scale


def quantize_to_levels(w, levels):
    indices = np.argmin(np.abs(w[:, None] - levels[None, :]), axis=1)
    return levels[indices]


w_int4 = quantize_to_levels(sample_weights, int4_levels)
w_nf4 = quantize_to_levels(sample_weights, nf4_quantiles)

int4_err = np.abs(sample_weights - w_int4).mean()
nf4_err = np.abs(sample_weights - w_nf4).mean()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.hist(
    sample_weights, bins=50, color="steelblue", alpha=0.7, density=True, label="Weights"
)
for l in int4_levels:
    ax1.axvline(l, color="coral", alpha=0.5, lw=0.8)
ax1.set_title("int4: 16 uniform levels (coral lines)")
ax1.legend()

ax2.hist(
    sample_weights, bins=50, color="steelblue", alpha=0.7, density=True, label="Weights"
)
for l in nf4_quantiles:
    ax2.axvline(l, color="mediumseagreen", alpha=0.5, lw=0.8)
ax2.set_title("NF4: 16 normal-distribution-quantile levels (green lines)")
ax2.legend()

plt.suptitle("NF4 places more levels where weights actually are (near zero)")
plt.tight_layout()
plt.show()

print(f"Mean absolute error on typical LLM weights (σ=0.02):")
print(f"  int4: {int4_err:.6f}")
print(
    f"  NF4:  {nf4_err:.6f}  ({(1 - nf4_err/int4_err)*100:.1f}% less error than int4)"
)
print()
print(
    "→ NF4 reduces quantization error by ~{:.0f}% for normal-distributed weights".format(
        (1 - nf4_err / int4_err) * 100
    )
)
print("  This is why QLoRA uses NF4 instead of regular int4:")
print("  `BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4')`")
print(
    "  (See learning/genai/04-llm/02-llm-finetuning-parameter-techniques.ipynb Section 6)"
)

---

## Part 7 — Summary and Closing Decision

We have worked through six quantization methods, all evaluated against Riverside's constraint: 16 GB MacBook, no internet, manuscript confidentiality. The table below maps what we learned to the original roadmap:

| Part | Concept     | Riverside answer                                          |
| ---- | ----------- | --------------------------------------------------------- |
| 1    | int8 basics | scale/zp rounding error is tiny per-weight; memory halves |
| 2    | Dynamic PTQ | perplexity delta < 0.5 — negligible quality impact        |
| 3    | Static PTQ  | calibration on literary corpus gains another ~0.1–0.3     |
| 4    | GPTQ        | int4 = 3.5 GB for 7B; +0.5–1.5 perplexity (acceptable)    |
| 5    | GGUF Q4_K_M | 4.1 GB, ~30 tok/s on M3, no Python needed                 |
| 6    | NF4         | explains QLoRA's base model storage; bridges to 04-llm    |


In [ ]:
# ── Closing Decision ──────────────────────────────────────────────────────────
print("=" * 60)
print("  CLOSING DECISION — Riverside MacBook Deployment")
print("=" * 60)
print()
print(
    f"  Constraint: MacBook {MACBOOK_VRAM_GB} GB unified memory (must leave ≥2 GB for system)"
)
print()
print("  Options evaluated:")
print(f"    bf16 (baseline): 14.0 GB  ✗ OOM  (only 2 GB headroom)")
print(f"    dynamic int8:     7.0 GB  ✓  (9 GB free, acceptable perplexity +0.1)")
print(f"    GPTQ int4:        3.5 GB  ✓  (12.5 GB free, perplexity +0.5–1.5)")
print(f"    GGUF Q4_K_M:      4.1 GB  ✓  (11.9 GB free, ~30 tok/s on M3)")
print()
print("  RECOMMENDATION: GGUF Q4_K_M via llama.cpp")
print("    - No Python environment needed on author MacBooks")
print("    - ~30 tokens/second on Apple M3 (responsive for editing assistant)")
print("    - Q4_K_M quality: +0.3 perplexity over bf16 (barely perceptible)")
print("    - Memory: 4.1 GB → 11.9 GB free for other apps")
print()
print("  ALTERNATIVE for developers: dynamic int8 via PyTorch")
print("    torch.quantization.quantize_dynamic(model, {nn.Linear}, dtype=torch.qint8)")
print("    5 lines of code, 7 GB, +0.1 perplexity")
print()
print("  RULE: Use the lightest quantization that keeps perplexity delta < 1.0")
print("  For literary editing: Q4_K_M stays well within that threshold.")

---

## What This Notebook Covered (and What It Didn't)

### Tier 1 — Implemented and Demonstrated

- int8 quantization math — scale, zero_point, rounding error measured on real weights
- Dynamic PTQ — applied to GPT-2; perplexity measured
- Static PTQ with calibration — calibrated on literary corpus
- GPTQ principles — algorithm explained; memory sizes computed for all model scales
- GGUF formats — Q4_K_M vs Q5_K_M vs Q8_0 comparison table
- NF4 — non-uniform levels demonstrated; error comparison vs. int4

### Tier 2 — Explained but Not Fully Built

- **AWQ (Activation-Aware Weight Quantization)** — similar to GPTQ but protects high-activation channels; referenced but not implemented (requires large calibration run)

### Tier 3 — Named but Out of Scope

- **SpQR** — sparse quantization; individual weight importance scores
- **SmoothQuant** — smooths outlier activations before quantization
- **QuIP** — incoherence processing for better int4 quality


---

## When to Use What

| Constraint                          | Method                           | GB saved | Quality delta    |
| ----------------------------------- | -------------------------------- | -------- | ---------------- |
| Developer laptop, PyTorch available | Dynamic int8                     | 50%      | Tiny (+0.1)      |
| Server inference, CUDA available    | GPTQ int4                        | 75%      | Small (+0.5–1.5) |
| End-user deployment, no Python      | GGUF Q4_K_M                      | 71%      | Small (+0.3)     |
| Training with limited VRAM          | QLoRA (NF4 base + bf16 adapters) | 75% base | Same as LoRA     |
| Maximum quality at any size         | bf16 or fp32                     | 0%       | Best             |

→ **Next:** `learning/ai-infrastructure/07-inference-systems/` — now that the model fits on the device, this chapter covers how to serve it efficiently: KV cache, continuous batching, speculative decoding.
